# Imports

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy import ndimage as ndi
from tqdm.auto import tqdm

from uavloc.data.annotated import IGNORE_LABEL
from uavloc.features.imagenet import IMAGENET_MEAN, IMAGENET_STD
from uavloc.segmentation.eval import (HOLDOUT_LOCATION, STRIP_LOCATION, STRIP_ROWS, bilinear_argmax, label_at_stride,
                                      load_baseline_model, load_val_sets, run_model)
from uavloc.segmentation.instances import BUILDING, building_table, components
from uavloc.segmentation.losses import ce_dice_loss, gap_band, gap_weighted_ce
from uavloc.track import REPO_ROOT, set_seed

sys.path.insert(0, str(REPO_ROOT / "pocs" / "village-seg-baseline"))
import village_seg as vs

# Params

In [ ]:
DATASET = "ua-village-annotated"
DEVICE = "cpu"
SEED = 42
GAP_RADIUS_PX = 10
SAT_CROP_PX = 512
SAT_BATCH = 4
RARE_CROP_P = 0.3
GAP_PX = 64
SHARE_BATCHES = 50
FIGURE_CROPS = 4
POC_MERGED = 595
POC_ORACLE_SPLIT = {"merged": 263, "separate": 329}
FIGURES = "experiments/019-building-gap-loss/figures"

In [ ]:
MAPS = ("stride-8 oracle, bilinear", "LR-ASPP")
set_seed(SEED)
(REPO_ROOT / FIGURES).mkdir(exist_ok=True)

# Data

In [ ]:
val_sets = load_val_sets(DATASET)

In [ ]:
sat = vs.load_satellite(REPO_ROOT / "data" / "derived" / DATASET, REPO_ROOT / "data" / "derived")
vs.ban_holdouts(sat, HOLDOUT_LOCATION, STRIP_LOCATION, STRIP_ROWS, GAP_PX)
sampler = vs.SatelliteSampler([s for s in sat if not s.banned.all()], SAT_CROP_PX, RARE_CROP_P)

# Model

In [ ]:
cases = run_model(load_baseline_model(DEVICE), val_sets, DEVICE)

# Run

In [ ]:
def case_table(name: str, case) -> pd.DataFrame:
    """`building_table` of one val case under the bilinear stride-8 oracle and LR-ASPP."""
    label = case.image.label
    maps = dict(zip(MAPS, (label_at_stride(label, "bilinear"), bilinear_argmax(case.output, case.image.rgb))))
    return building_table(label, maps).assign(set=name, date=case.image.date)

In [ ]:
buildings = pd.concat([case_table(name, c) for name, items in cases.items() for c in items], ignore_index=True)
len(buildings)

In [ ]:
merged = buildings[buildings["LR-ASPP"] == "merged"]
oracle_split = merged["stride-8 oracle, bilinear"].value_counts().to_dict()
len(merged), oracle_split

In [ ]:
assert len(merged) == POC_MERGED and all(oracle_split.get(k) == v for k, v in POC_ORACLE_SPLIT.items())

In [ ]:
rng = np.random.default_rng(SEED)
x, y = vs.batch_of(sampler, rng, SAT_BATCH)
band = gap_band(y, GAP_RADIUS_PX)

In [ ]:
def reference_band(label: np.ndarray, radius_px: int) -> np.ndarray:
    """Independent `gap_band`: labelled non-building pixels covered by the square dilations of >= 2 buildings."""
    comp, n = components(label == BUILDING)
    square = np.ones((2 * radius_px + 1,) * 2, bool)
    count = sum(ndi.binary_dilation(comp == k, square).astype(int) for k in range(1, n + 1))
    return (np.asarray(count) >= 2) & (label != IGNORE_LABEL) & (label != BUILDING)

In [ ]:
reference = np.stack([reference_band(crop, GAP_RADIUS_PX) for crop in y.numpy()])
band_matches = bool((reference == band.numpy()).all())
on_building_or_ignored = int((band & ((y == BUILDING) | (y == IGNORE_LABEL))).sum())
band_matches, on_building_or_ignored

In [ ]:
model = load_baseline_model(DEVICE).train()
with torch.no_grad():
    logits = model(x)
loss_diff = abs(gap_weighted_ce(logits, y, band, 0.0).item() - ce_dice_loss(logits, y).item())
loss_diff

In [ ]:
assert band_matches and on_building_or_ignored == 0 and loss_diff < 1e-5

In [ ]:
band_px, labelled_px = 0, 0
for _ in tqdm(range(SHARE_BATCHES), desc="band share"):
    _, yb = vs.batch_of(sampler, rng, SAT_BATCH)
    band_px += int(gap_band(yb, GAP_RADIUS_PX).sum())
    labelled_px += int((yb != IGNORE_LABEL).sum())
band_share = band_px / labelled_px
band_share

# Plots

In [ ]:
def band_overlay(x_crop: torch.Tensor, y_crop: torch.Tensor, band_crop: torch.Tensor) -> np.ndarray:
    """RGB crop with buildings tinted red and the gap band in cyan."""
    rgb = (x_crop.numpy().transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN).clip(0, 1)
    out = rgb.copy()
    out[y_crop.numpy() == BUILDING] = 0.5 * rgb[y_crop.numpy() == BUILDING] + [0.5, 0, 0]
    out[band_crop.numpy()] = [0, 1, 1]
    return out

In [ ]:
fig_band, axes = plt.subplots(1, FIGURE_CROPS, figsize=(4.5 * FIGURE_CROPS, 4.8))
for k, ax in enumerate(axes):
    ax.imshow(band_overlay(x[k], y[k], band[k])); ax.axis("off")
    ax.set_title(f"band {band[k].float().mean():.1%} of crop", fontsize=9)
fig_band.suptitle(f"gap band (cyan), radius {GAP_RADIUS_PX} px, buildings red")
fig_band.tight_layout()
fig_band.savefig(REPO_ROOT / FIGURES / "gap-band-crops.png", dpi=110, bbox_inches="tight")